# ЛИНЕЙКА ЖЕСТКАЯ

## IMPORT

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler, OneHotEncoder
import json
from pyspark.sql.functions import col,when
import os
import shutil
os.environ['HADOOP_HOME'] = "C:\\hadoop" 
# Добавляем путь к bin в системный PATH
os.environ['PATH'] += os.pathsep + "C:\\hadoop\\bin"

## СОЗДАНИЕ СЕССИИ

In [8]:
spark = SparkSession.builder.appName('Logistic_reg').getOrCreate()

## LOAD DOTA

In [ ]:
# Load training data
train_data_path = '../datasets/joined/train_data.parquet'
features_path = '../datasets/joined/columns_list.json'
training = spark.read.parquet(train_data_path)

with open(features_path, 'r') as file:
        cols = json.load(file)

# СОЗДАЛ КОЛОНКИ
feature_cols = cols['all']
cat_cols = cols['cat']
numeric_cols = list(set(feature_cols) - set(cat_cols))

for cat in cat_cols:
        training = training.withColumn(cat,col(cat)+1)

# ВХОДНЫЕ КАТЕГОРИАЛЬНЫЕ КОЛОНКИ
to_encode_in = cat_cols
to_encode_out = [c+"_vec" for c in cat_cols]

# ОБЪЯВИЛ ЭНКОДЕР И ОБРАБОТАЛ
encoder = OneHotEncoder(inputCols=to_encode_in,
                        outputCols=to_encode_out,
                        handleInvalid='keep')  # Вот это спасет от ошибки)
training_encoded = encoder.fit(training).transform(training)

# ВЕКТОРИЗОВАЛ
assembler_inputs = to_encode_out + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

final_training = assembler.transform(training_encoded) 

# Define the model
lr = LogisticRegression(
    featuresCol='features',
    labelCol='target'
)

# Create parameter grid
paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.001, 0.01, 0.1])
             .addGrid(lr.elasticNetParam, [0.0, 0.5])
             .addGrid(lr.maxIter, [20, 50])
             .build())

# Define evaluator
evaluator = BinaryClassificationEvaluator(
    rawPredictionCol='rawPrediction',
    labelCol='target',
    metricName='areaUnderROC'
)

sample_fraction = 0.3
print(f"Sampling {sample_fraction*100}% of data for grid search...")
training_sample = final_training.sample(False, sample_fraction, seed=42)
training_sample.cache()
print(f"Training sample count: {training_sample.count()}")

# CrossValidator with sampling for big data
# For production, you might want to use 3-5 folds
crossval = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,  # Adjust based on your cluster
    seed=42
)

print("Starting grid search...")
cvModel = crossval.fit(training_sample)

best_lr_model = cvModel.bestModel
best_reg_param = best_lr_model.getRegParam()
best_elastic_net = best_lr_model.getElasticNetParam()
best_max_iter = best_lr_model.getMaxIter()
sample_predictions = best_lr_model.transform(training_sample)
sample_auc = evaluator.evaluate(sample_predictions)
print(f"Sample AUC: {sample_auc:.4f}")

training_sample.unpersist()

In [10]:
def finalize_csv(folder_path, target_name):
    # 1. Проверяем, существует ли папка
    if not os.path.exists(folder_path):
        print(f"Папка {folder_path} не найдена")
        return

    files = os.listdir(folder_path)
    
    # 2. Ищем файл данных (обычно начинается на part- и заканчивается на .csv)
    # Важно: Spark создает скрытые файлы ._SUCCESS и .crc, их игнорируем
    part_files = [f for f in files if f.startswith("part-") and f.endswith(".csv")]
    
    if not part_files:
        print("Файл с данными не найден в папке")
        return
        
    part_file = part_files[0]
    
    # 3. Формируем путь назначения (в ту же папку, где лежала папка-результат)
    parent_dir = os.path.dirname(os.path.abspath(folder_path))
    final_path = os.path.join(parent_dir, target_name)
    
    # 4. Переносим файл (shutil.move надежнее os.rename между разделами/дисками)
    temp_full_path = os.path.join(folder_path, part_file)
    
    if os.path.exists(final_path):
        os.remove(final_path)
        
    shutil.move(temp_full_path, final_path)
    
    # 5. Удаляем всю временную папку целиком (вместе со скрытыми файлами .crc)
    shutil.rmtree(folder_path)
    print(f"Готово! Файл сохранен как: {final_path}")

## ЗАКАЧКА ТЕСТА

In [ ]:
# Create new model with best parameters
best_lr = LogisticRegression(
    featuresCol='features',
    labelCol='target',
    regParam=best_reg_param,
    elasticNetParam=best_elastic_net,
    maxIter=best_max_iter
)

# Train on FULL dataset
final_model = best_lr.fit(final_training)
print("Full model training complete!")

In [11]:
#  Load test data
test_data_path = '../datasets/joined/test_data.parquet'
testing = spark.read.parquet(test_data_path)

testing = testing.withColumn("was_hacked", 
    when(col("was_hacked") >0, 1).cast("byte")       # В остальных случаях пытаемся в число
).fillna(-1, subset=["was_hacked"])                  # Все пустые/невалидные -> -1
for cat in cat_cols:
        testing = testing.withColumn(cat,col(cat)+1)
        
# ОБЪЯВИЛ ЭНКОДЕР И ОБРАБОТАЛ
test_encoded = encoder.fit(training).transform(testing)

final_test = assembler.transform(test_encoded) 


## ДУРАЦКАЯ РАБОТА С ПАПКАМИ

In [ ]:
from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

# 1. Трансформируем тестовые данные
predictions = best_lr_model.transform(final_test)

# 2. Достаем вероятность КЛАССА 1 (фрод) из вектора [p0, p1]
# vector_to_array превращает вектор в массив, [1] берет второй элемент
y_out = predictions.withColumn("prob_array", vector_to_array("probability")) \
                   .select(
                       F.col("event_id"), 
                       F.col("prob_array")[1].alias("predict")
                   )

# 3. Сохраняем во временную ПАПКУ
temp_path = '../submit/temp_folder'
y_out.coalesce(1).write.csv(
    temp_path,
    header=True,
    mode='overwrite'
)

# 4. Вызываем твою функцию (убедись, что она импортирована)
# Первый аргумент - ПАПКА, которую создал Spark
# Второй аргумент - ИМЯ итогового файла
finalize_csv(temp_path, "lr_submit_1.csv")


Готово! Файл сохранен как: c:\antifraud_hak\antifraud_hak\submit\lr_submit_1.csv
